**RAG as a sequential pipeline**

I outlined the complete lifecycle of an advanced RAG (Retrieval-Augmented Generation) Pipeline  

1. Preprocessing

2. Index Construction

3. LLM-Augmented Retrieval

4. Evaluation

**Evaluating the final output**

I outlined the fundamental pillars to evaluate the results

4.1 Retrieval Quality

4.2 Generation Quality

4.3 End-to-End QA Accuracy

**3. LLM-AUGMENTED RETRIEVAL**

**3.1 Baseline LLM**

A smoker presents with painless hematuria

The question is justified to contains related context such as
- urothelial carcinoma
- transitional cell carcinoma
- bladder cancer

***3.1.1 Prompting***

In [106]:
QUERY_PROMPT = """
You are a medical retrieval specialist.

Given a USMLE question,
generate five search queries.

Include:

1. Disease
2. Pathophysiology
3. Histology
4. Pharmacology
5. Differential diagnosis

Return JSON only.

Question:
{question}
"""

*****3.1.2 Injected Knowledge-Based Prompting*****

***Expected Results when running baseline LLM***

**Input**

In [107]:
"45-year-old diabetic woman with nephrotic syndrome"

'45-year-old diabetic woman with nephrotic syndrome'

**Output**

In [108]:
["diabetic nephropathy nephrotic syndrome",
 "nodular glomerulosclerosis",
 "Kimmelstiel Wilson lesion",
 "proteinuria diabetes mellitus",
 "renal pathology diabetic kidney disease"]

['diabetic nephropathy nephrotic syndrome',
 'nodular glomerulosclerosis',
 'Kimmelstiel Wilson lesion',
 'proteinuria diabetes mellitus',
 'renal pathology diabetic kidney disease']

In [ ]:
from google import genai
import json

client = genai.Client(api_key="os.environ["GEMINI_API_KEY"]")


In [ ]:
def expand_query(question):

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=QUERY_PROMPT.format(question=question))

    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    data = json.loads(text)

    return [item["query"] for item in data["search_queries"]]

In [111]:
queries = expand_query("23-year-old pregnant woman with dysuria at 22 weeks gestation")

print(queries)

['Urinary tract infection in pregnant women', 'Pathophysiology of UTIs during pregnancy', 'Histology of bladder inflammation in cystitis', 'Safe antibiotics for dysuria in 22-week pregnant woman', 'Differential diagnosis of dysuria in pregnancy']


***3.2 Hybrid Search***

***3.2.1 FAISS vector on Hybrid Search***

I defined numbers of relevant search results, normalized the embeddings, and forced the Python array into the exact format

In [112]:
!pip install -q faiss-cpu

In [113]:
import faiss

In [114]:
faiss_index = faiss.read_index("/kaggle/input/datasets/saridacharoenngampit/faiss-index1/faiss_index.bin")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


**Notes:**  UNEXPECTED is a known benign mismatch that commonly appears after Transformers-version changes. Since position_ids is typically a non-trainable buffer, ignoring it usually has no effect on inference or fine-tuning. Source: https://discuss.huggingface.co/t/embeddings-position-ids-unexpected-warning-started-showing/173102

In [116]:
import numpy as np
def dense_search(query, top_k=10): 
    q_emb = embedding_model.encode([query],normalize_embeddings=True)
    D, I = faiss_index.search(np.array(q_emb).astype("float32"),top_k)
    return [{"id": int(i)} for i in I[0]]

In [117]:
print(dense_search("urinary tract infection in pregnancy"))

[{'id': 7875}, {'id': 5660}, {'id': 0}, {'id': 6695}, {'id': 9620}, {'id': 5445}, {'id': 1350}, {'id': 7617}, {'id': 1629}, {'id': 5723}]


***3.2.2 BM25 vector on Hybrid Search***

 I defined numbers of relevant search results, normalized the embeddings,
 set up one score for one document, sorted the scores from lowest to highest, and forced the Python array into the exact format

In [118]:
!pip install -q rank-bm25

In [119]:
from rank_bm25 import BM25Okapi

In [120]:
import pickle

file_path = "/kaggle/input/datasets/saridacharoenngampit/tokenized/tokenized_docs-2.pkl"

with open(file_path, "rb") as f:
    tokenized_docs = pickle.load(f)

In [121]:
bm25 = BM25Okapi(tokenized_docs)

In [122]:
def sparse_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [{"id": int(i), "score": float(scores[i]), "tokenized_docs": tokenized_docs[i] }  for i in top_idx]

In [123]:
def show_results(results, k=5):
    for r in results[:k]:
        print(f"ID: {r['id']} | Score: {r['score']:.3f}")
        print("Preview:", " ".join(r["tokenized_docs"][:15]))
        print()

show_results(sparse_search("urinary tract infection in pregnancy"))

ID: 6695 | Score: 19.592
Preview: question: one day after giving birth to a 4050-g (8-lb 15-oz) male newborn, a 22-year-old

ID: 5660 | Score: 18.566
Preview: question: a 24-year-old woman calls her gynecologist complaining of vaginal odor and vaginal discharge. she

ID: 5260 | Score: 18.432
Preview: question: a 64-year-old male retired farmer presents to the orthopaedic surgery clinic with chronic left

ID: 7392 | Score: 16.769
Preview: question: a 32-year-old man with hypertension and gout comes to the physician with left flank

ID: 2671 | Score: 16.402
Preview: question: a 35-year-old woman comes to the physician because of progressive left flank pain and



***3.3 Combination of two vectors as Hybrid & Multiple Queries***

***3.3.1 Multiple Queries***

I rewrited a question to catch different phrasings and matched semantic (dense_search) and keyword (sparse_search)

***Expected Results when running a retrieval***

**Input**

In [124]:
"Query"

'Query'

**Output**

In [125]:
["Query", 
    "Expanded Query docs 1", 
    "Expanded Query docs 2", 
    "Expanded Query docs 3", 
    "Expanded Query docs 4", 
    "Expanded Query docs 5"]

['Query',
 'Expanded Query docs 1',
 'Expanded Query docs 2',
 'Expanded Query docs 3',
 'Expanded Query docs 4',
 'Expanded Query docs 5']

In [126]:
def expand_query(question):
    return []

In [127]:
def retrieve_multi_query(question):
    expanded = expand_query(question)
    all_queries = [question] + expanded
    retrieval_sets = []
    for q in all_queries:
        dense_docs = dense_search(q)
        sparse_docs = sparse_search(q)
        retrieval_sets.append(dense_docs + sparse_docs)
    return retrieval_sets

In [128]:
results = retrieve_multi_query("23-year-old pregnant woman with dysuria at 22 weeks gestation")

In [129]:
for query_num, docs in enumerate(results, 1):
    print(f"\n Query {query_num}")


 Query 1


In [145]:
for doc_num, doc in enumerate(docs[:5], 1):
    print(f"\nDocument {doc_num}")
    print(doc)


Document 1
{'id': 537}

Document 2
{'id': 6799}

Document 3
{'id': 6391}

Document 4
{'id': 5627}

Document 5
{'id': 8782}


***3.3.2 Combination of two vectors as Hybrid search***

I created a list of search results, set up a score tracker, calculated the fractional score, and sorted in descending order

In [131]:
from collections import defaultdict

In [154]:
def reciprocal_rank_fusion(retrieval_sets, k=60):
    scores = defaultdict(float)
    for docs in retrieval_sets:
        for rank, doc in enumerate(docs):
            if isinstance(doc, (int, np.integer)):
                doc_id = int(doc)
            elif isinstance(doc, dict):
                doc_id = int(doc["id"])
            else:
                continue
            scores[doc_id] += 1.0 / (k + rank + 1)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [133]:
retrieval_sets = retrieve_multi_query("23-year-old pregnant woman with dysuria at 22 weeks gestation")

In [ ]:
fused_results = reciprocal_rank_fusion(retrieval_sets)

In [ ]:
ranked_docs = [doc_id for doc_id, _ in fused_results[:5]]

In [155]:
for doc_id, score in fused_results[:10]:
    print(f"Doc ID: {doc_id} | Score: {score:.4f}")

Doc ID: 537 | Score: 0.0164
Doc ID: 6799 | Score: 0.0161
Doc ID: 6391 | Score: 0.0159
Doc ID: 5627 | Score: 0.0156
Doc ID: 8782 | Score: 0.0154
Doc ID: 8907 | Score: 0.0152
Doc ID: 3651 | Score: 0.0149
Doc ID: 3018 | Score: 0.0147
Doc ID: 2753 | Score: 0.0145
Doc ID: 9147 | Score: 0.0143


***3.3.3 Combination of two vectors as Hybrid search***

I created a list of structural pairs and sorted them in descending order

In [ ]:
 from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

**Notes:**  UNEXPECTED is a known benign mismatch that commonly appears after Transformers-version changes. Since position_ids is typically a non-trainable buffer, ignoring it usually has no effect on inference or fine-tuning. Source: https://discuss.huggingface.co/t/embeddings-position-ids-unexpected-warning-started-showing/173102

In [157]:
documents = [ {"id": i, "text": " ".join(tokenized_docs[i])}
    for i in range(len(tokenized_docs))]

In [158]:
def rerank(question,
    candidates):
    pairs = []
    for doc_id,_ in candidates:
        pairs.append([question,documents[doc_id]["text"]])
    
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates,scores), key=lambda x:x[1],reverse=True)
    return ranked

In [165]:
question = "urinary tract infection in pregnancy"

In [167]:
ranked_results = rerank(question, fused_results)
for i, ((doc_id, rrf_score), rerank_score) in enumerate(ranked_results[:5], 1):
    print(f"\nRank {i}")
    print("Doc ID:", doc_id)
    print("RRF:", rrf_score)
    print("Rerank:", rerank_score)
    print("Text:", documents[doc_id]["text"][:200])


Rank 1
Doc ID: 9147
RRF: 0.014285714285714285
Rerank: -3.5386057
Text: question: a 7-year-old girl is brought to the physician with complaints of recurrent episodes of dysuria for the past few months. her parents reported 4 to 5 similar episodes in the last year. they al

Rank 2
Doc ID: 537
RRF: 0.01639344262295082
Rerank: -4.1004553
Text: question: a 35-year-old woman, gravida 2, para 1, at 16 weeks' gestation comes to the office for a prenatal visit. she reports increased urinary frequency but otherwise feels well. pregnancy and deliv

Rank 3
Doc ID: 3651
RRF: 0.014925373134328358
Rerank: -4.7703967
Text: question: a 25-year-old woman presents to the clinic with complaints of dysuria and increased urinary frequency. her urinalysis results are negative for nitrites. urine microscopy shows the findings i

Rank 4
Doc ID: 8907
RRF: 0.015151515151515152
Rerank: -5.071681
Text: question: a 36-year-old primigravida woman visits her gynecologist during the 28th week of her pregnancy. physic